In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent.parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from IPython.display import display, Markdown

from ocr_vs_vlm.results_final.shared.colors import get_model_color, APPROACH_COLORS
from ocr_vs_vlm.results_final.shared.stats_utils import (
    bootstrap_ci, wilcoxon_test, cohens_d, effect_size_interpretation
)
from ocr_vs_vlm.results_final.shared.viz_utils import setup_plotly_template
from ocr_vs_vlm.results_final.shared.data_loader import (
    load_dataset_data, extract_models_from_columns
)

setup_plotly_template()

DATASET = 'VOC2007'
print(f"✓ Setup complete for {DATASET}")

✓ Setup complete for VOC2007


## 1. Load Data

In [2]:
try:
    df = load_dataset_data(DATASET, task_type='parsing')
    print(f"✓ Loaded {len(df)} samples")
    print(f"  Approaches: {df['approach'].unique().tolist()}")
except Exception as e:
    print(f"Error loading data: {e}")
    df = pd.DataFrame()

models = extract_models_from_columns(df) if len(df) > 0 else []
print(f"  Models: {models}")

Error loading data: load_dataset_data() got an unexpected keyword argument 'task_type'
  Models: []


## 2. Performance Summary

In [3]:
# Compute stats
stats = []
for approach in df['approach'].unique() if len(df) > 0 else []:
    app_df = df[df['approach'] == approach]
    for model in models:
        col = f'cer_{model}'
        if col in app_df.columns:
            scores = app_df[col].dropna().values
            if len(scores) > 0:
                mean, ci_lo, ci_hi = bootstrap_ci(scores, 'mean')
                stats.append({
                    'Approach': approach,
                    'Model': model,
                    'CER': mean,
                    'CI_Lower': ci_lo,
                    'CI_Upper': ci_hi,
                    'N': len(scores)
                })

stats_df = pd.DataFrame(stats)
if len(stats_df) > 0:
    display(Markdown("### CER Summary (Lower is Better)"))
    pivot = stats_df.pivot_table(index='Model', columns='Approach', values='CER').round(4)
    display(pivot)

In [4]:
# Visualization
if len(stats_df) > 0:
    fig = go.Figure()
    
    stats_sorted = stats_df.sort_values('CER')
    
    fig.add_trace(go.Bar(
        x=[f"{row['Model']} ({row['Approach']})" for _, row in stats_sorted.iterrows()],
        y=stats_sorted['CER'],
        error_y=dict(
            type='data',
            array=stats_sorted['CI_Upper'] - stats_sorted['CER'],
            arrayminus=stats_sorted['CER'] - stats_sorted['CI_Lower']
        ),
        marker_color=[get_model_color(m) for m in stats_sorted['Model']],
        text=stats_sorted['CER'].round(3),
        textposition='outside'
    ))
    
    fig.update_layout(
        title=f'{DATASET}: Character Error Rate (CER ↓)',
        yaxis_title='CER',
        height=500,
        xaxis_tickangle=-45
    )
    fig.show()

## 3. Statistical Analysis

In [5]:
# Approach comparison
if len(df) > 0:
    approach_scores = {}
    
    for approach in df['approach'].unique():
        app_df = df[df['approach'] == approach]
        all_scores = []
        for model in models:
            col = f'cer_{model}'
            if col in app_df.columns:
                all_scores.extend(app_df[col].dropna().tolist())
        approach_scores[approach] = np.array(all_scores)
    
    approaches = list(approach_scores.keys())
    if len(approaches) >= 2:
        a1, a2 = approaches[0], approaches[1]
        if len(approach_scores[a1]) > 0 and len(approach_scores[a2]) > 0:
            n = min(len(approach_scores[a1]), len(approach_scores[a2]))
            stat, p = wilcoxon_test(approach_scores[a1][:n], approach_scores[a2][:n])
            d = cohens_d(approach_scores[a1], approach_scores[a2])
            
            m1, m2 = np.mean(approach_scores[a1]), np.mean(approach_scores[a2])
            winner = a1 if m1 < m2 else a2
            
            print(f"📊 {a1} vs {a2}")
            print(f"   Mean CER: {m1:.4f} vs {m2:.4f}")
            print(f"   Winner: {winner}")
            print(f"   p-value: {p:.4f} | d={d:.3f} ({effect_size_interpretation(d)})")

## 4. Key Findings

In [6]:
print("=" * 60)
print(f"{DATASET} KEY FINDINGS")
print("=" * 60)

if len(stats_df) > 0:
    best = stats_df.loc[stats_df['CER'].idxmin()]
    print(f"\n🏆 Best: {best['Approach']} + {best['Model']}: CER = {best['CER']:.4f}")
    
    approach_means = stats_df.groupby('Approach')['CER'].mean().sort_values()
    print(f"\n📊 Approach Ranking:")
    for i, (app, score) in enumerate(approach_means.items(), 1):
        print(f"   {i}. {app}: {score:.4f}")

print("\n" + "=" * 60)

VOC2007 KEY FINDINGS

